In [ ]:

import InversionNet
import cddpm
import numpy as np
import torch
import matplotlib.pyplot as plt
import load
import tddpm
import gan


In [ ]:
S_dataset = load.SMILE_dataset()
cddpm_class = cddpm.cddpm()
tddpm_class = tddpm.tddpm()
# tddpm_class2 = tddpm.tddpm()
InvNet_class = InversionNet.InversionNet()
Gan_class = gan.Gan()


In [ ]:
cddpm_class.train(S_dataset, "FM", save_name="cFM", exclude = [2, 7])

cddpm_class.train(S_dataset, "DDPM", save_name="cDDPM", exclude = [2, 7])

In [ ]:
tddpm_class.train(S_dataset, "FM", save_name="EIFM", exclude = [2,7])
tddpm_class.train(S_dataset, "DDPM", save_name="EIDDPM", exclude = [2,7])


In [ ]:
ckpt_Gan = None
Gan_class.train(S_dataset, save_name="Gan", exclude = [2,7], init_ckpt=ckpt_Gan)

In [ ]:
ckpt_InvNet = None
InvNet_class.train(S_dataset, save_name="InvNet", exclude = [2,7], init_ckpt=ckpt_InvNet)

In [ ]:
tddpm_class2 = tddpm.tddpm()
cddpm_class2 = cddpm.cddpm()


ckpt_tFM = torch.load("saved_model/tFM_1000.pth", weights_only=True)
ckpt_tDDPM = torch.load("saved_model/tDDPM_1000.pth", weights_only=True)
ckpt_cFM = torch.load("saved_model/cFM_1000.pth", weights_only=True)
ckpt_cDDPM = torch.load("saved_model/cDDPM_1000.pth", weights_only=True)
ckpt_InvNet = torch.load("saved_model/InvNet_1000.pth", weights_only=True)
ckpt_Gan = torch.load("saved_model/Gan_g_1000.pth", weights_only=True)

tddpm_class.denoise_model.load_state_dict(ckpt_tFM)
tddpm_class2.denoise_model.load_state_dict(ckpt_tDDPM)
cddpm_class2.denoise_model.load_state_dict(ckpt_cDDPM)
cddpm_class.denoise_model.load_state_dict(ckpt_cFM)
InvNet_class.supervised_model.load_state_dict(ckpt_InvNet)
Gan_class.model.load_state_dict(ckpt_Gan)

In [ ]:
# tddpm_class.denoise_model(a,b,t).shape

In [ ]:

@torch.no_grad()
def ssim_batch(
    x,
    y,
    data_range: float = S_dataset.dr,
    reduction: str = "mean",
    device: torch.device | str | None = None,
    dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """
    Compute SSIM between two batches of images.

    Args:
        x, y: torch.Tensor or np.ndarray of shape (B, 1, 32, 32).
        data_range: If None, inferred from (x,y). Otherwise provide explicitly (e.g., 1.0).
        reduction: "mean" or "none".
        device: Device to run the metric on (e.g., "cuda").
        dtype: Tensor dtype after conversion.

    Returns:
        If reduction == "mean": scalar tensor.
        If reduction == "none": tensor of shape (B,) with per-image SSIM.
    """
    # Convert numpy inputs to torch
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x)
    if isinstance(y, np.ndarray):
        y = torch.from_numpy(y)

    if not (isinstance(x, torch.Tensor) and isinstance(y, torch.Tensor)):
        raise TypeError("x and y must be torch.Tensor or np.ndarray.")
    if x.shape != y.shape:
        raise ValueError(f"Shape mismatch: x {tuple(x.shape)} vs y {tuple(y.shape)}.")
    if x.ndim != 4 or x.shape[1] != 1:
        raise ValueError(f"Expected shape (B,1,H,W). Got {tuple(x.shape)}.")
    if reduction not in {"mean", "none"}:
        raise ValueError('reduction must be "mean" or "none".')

    # Device handling
    if device is None:
        device = x.device
    device = torch.device(device)

    x = x.to(device=device, dtype=dtype)
    y = y.to(device=device, dtype=dtype)

    if reduction == "mean":
        metric = SSIM(data_range=data_range).to(device)
        return metric(x, y)
    else:
        vals = []
        for i in range(x.shape[0]):
            metric_i = SSIM(data_range=data_range).to(device)
            vals.append(metric_i(x[i:i+1], y[i:i+1]))
        return torch.stack(vals, dim=0).view(-1)

In [ ]:
# test once


repnum = 1

k=0
# for j in range(7,8):
for j in range(0,10):
    print(j)

    MSE_avg_tFM_G = []
    MSE_avg_cFM_G = []
    MSE_avg_InvNet_G = []
    MSE_avg_Gan_G = []
    MSE_avg_tDDPM_G = []
    MSE_avg_cDDPM_G = []

    MAE_avg_tFM_G = []
    MAE_avg_cFM_G = []
    MAE_avg_InvNet_G = []
    MAE_avg_Gan_G = []
    MAE_avg_tDDPM_G = []
    MAE_avg_cDDPM_G = []

    SSIM_avg_tFM_G = []
    SSIM_avg_cFM_G = []
    SSIM_avg_InvNet_G = []
    SSIM_avg_Gan_G = []
    SSIM_avg_tDDPM_G = []
    SSIM_avg_cDDPM_G = []

    for h in range(repnum):
        x_test, y_test = S_dataset.get_test_data(64, [j])

        tFM_res = tddpm_class.sampler_FM(tddpm_class.denoise_model, y_test, 200)
        tDDPM_res = tddpm_class.sampler(tddpm_class2.denoise_model, y_test)
        cFM_res = cddpm_class.sampler_FM(cddpm_class.denoise_model, y_test, 200)
        cDDPM_res = cddpm_class.sampler(cddpm_class2.denoise_model, y_test)
        with torch.no_grad():
            InvNet_res = InvNet_class.supervised_model(y_test)
            Gan_res = Gan_class.model(y_test)


        tFM_res_cpu = (tFM_res).cpu().numpy()
        cFM_res_cpu = (cFM_res).cpu().numpy()
        InvNet_res_cpu = (InvNet_res).cpu().numpy()
        Gan_res_cpu = (Gan_res).cpu().numpy()
        tDDPM_res_cpu = (tDDPM_res).cpu().numpy()
        cDDPM_res_cpu = (cDDPM_res).cpu().numpy()

        x_test_cpu = x_test.cpu().numpy()

        MSE_avg_tFM = np.mean((tFM_res_cpu-x_test_cpu)**2)
        MSE_avg_cFM = np.mean((cFM_res_cpu-x_test_cpu)**2)
        MSE_avg_InvNet = np.mean((InvNet_res_cpu-x_test_cpu)**2)
        MSE_avg_Gan = np.mean((Gan_res_cpu-x_test_cpu)**2)
        MSE_avg_tDDPM = np.mean((tDDPM_res_cpu-x_test_cpu)**2)
        MSE_avg_cDDPM = np.mean((cDDPM_res_cpu-x_test_cpu)**2)
        
        MAE_avg_tFM = np.mean(np.abs(tFM_res_cpu-x_test_cpu))
        MAE_avg_cFM = np.mean(np.abs(cFM_res_cpu-x_test_cpu))
        MAE_avg_InvNet = np.mean(np.abs(InvNet_res_cpu-x_test_cpu))
        MAE_avg_Gan = np.mean(np.abs(Gan_res_cpu-x_test_cpu))
        MAE_avg_tDDPM = np.mean(np.abs(tDDPM_res_cpu-x_test_cpu))
        MAE_avg_cDDPM = np.mean(np.abs(cDDPM_res_cpu-x_test_cpu))
        
        MSE_avg_tFM_G.append(MSE_avg_tFM)
        MSE_avg_cFM_G.append(MSE_avg_cFM)
        MSE_avg_InvNet_G.append(MSE_avg_InvNet)
        MSE_avg_Gan_G.append(MSE_avg_Gan)
        MSE_avg_tDDPM_G.append(MSE_avg_tDDPM)
        MSE_avg_cDDPM_G.append(MSE_avg_cDDPM)
        
        MAE_avg_tFM_G.append(MAE_avg_tFM)
        MAE_avg_cFM_G.append(MAE_avg_cFM)
        MAE_avg_InvNet_G.append(MAE_avg_InvNet)
        MAE_avg_Gan_G.append(MAE_avg_Gan)
        MAE_avg_tDDPM_G.append(MAE_avg_tDDPM)
        MAE_avg_cDDPM_G.append(MAE_avg_cDDPM)

        # SSIM_avg_tFM = ssim_batch(tFM_res_cpu, x_test_cpu, reduction="none")
        # SSIM_avg_cFM = ssim_batch(cFM_res_cpu, x_test_cpu, reduction="none")
        #
        SSIM_avg_tFM = ssim_batch(tFM_res_cpu, x_test_cpu, reduction="mean")
        SSIM_avg_cFM = ssim_batch(cFM_res_cpu, x_test_cpu, reduction="mean")
        SSIM_avg_InvNet = ssim_batch(InvNet_res_cpu, x_test_cpu, reduction="mean")
        SSIM_avg_Gan = ssim_batch(Gan_res_cpu, x_test_cpu, reduction="mean")
        SSIM_avg_tDDPM = ssim_batch(tDDPM_res_cpu, x_test_cpu, reduction="mean")
        SSIM_avg_cDDPM = ssim_batch(cDDPM_res_cpu, x_test_cpu, reduction="mean")
        
        SSIM_avg_cFM_G.append(SSIM_avg_cFM)
        SSIM_avg_tFM_G.append(SSIM_avg_tFM)
        SSIM_avg_InvNet_G.append(SSIM_avg_InvNet)
        SSIM_avg_Gan_G.append(SSIM_avg_Gan)
        SSIM_avg_tDDPM_G.append(SSIM_avg_tDDPM)
        SSIM_avg_cDDPM_G.append(SSIM_avg_cDDPM)


    print("cFM, MSE="+str(np.mean(MSE_avg_cFM_G))+',MAE='+str(np.mean(MAE_avg_cFM_G))+' ,SSIM='+str(np.mean(SSIM_avg_cFM_G)))
    print("cDDPM, MSE="+str(np.mean(MSE_avg_cDDPM_G))+',MAE='+str(np.mean(MAE_avg_cDDPM_G))+' ,SSIM='+str(np.mean(SSIM_avg_cDDPM_G)))
    print("tFM, MSE="+str(np.mean(MSE_avg_tFM_G))+',MAE='+str(np.mean(MAE_avg_tFM_G))+' ,SSIM='+str(np.mean(SSIM_avg_tFM_G)))
    print("tDDPM, MSE="+str(np.mean(MSE_avg_tDDPM_G))+',MAE='+str(np.mean(MAE_avg_tDDPM_G))+' ,SSIM='+str(np.mean(SSIM_avg_tDDPM_G)))

    print("InvNet, MSE="+str(np.mean(MSE_avg_InvNet_G))+',MAE='+str(np.mean(MAE_avg_InvNet_G))+' ,SSIM='+str(np.mean(SSIM_avg_InvNet_G)))
    print("Gan, MSE="+str(np.mean(MSE_avg_Gan_G))+',MAE='+str(np.mean(MAE_avg_Gan_G))+' ,SSIM='+str(np.mean(SSIM_avg_Gan_G)))

    k+=1



In [ ]:
# visualization for selected classes

ind = np.random.choice(64)
print(ind)
cFM_ind = cFM_res_cpu[ind,0]
tFM_ind = tFM_res_cpu[ind,0]
InvNet_ind = InvNet_res_cpu[ind,0]

fig, axes = plt.subplots(1, 4)
x_truth = x_test_cpu[ind,0]
axes[0].imshow(cFM_ind)
MSE_cFM = np.mean((cFM_ind - x_truth)**2)
SSIM_cFM = ssim_batch(cFM_ind[np.newaxis,np.newaxis,:], x_truth[np.newaxis,np.newaxis,:]).cpu().numpy()
axes[0].title.set_text("cFM, MSE="+str(np.round(MSE_cFM,4))+'\n ,SSIM='+str(np.round(SSIM_cFM,4)))
# plt.show()
axes[1].imshow(tFM_ind)
MSE_tFM = np.mean((tFM_ind - x_truth)**2)
SSIM_tFM = ssim_batch(tFM_ind[np.newaxis,np.newaxis,:], x_truth[np.newaxis,np.newaxis,:]).cpu().numpy()
axes[1].title.set_text("tFM, MSE="+str(np.round(MSE_tFM,4))+'\n, SSIM='+str(np.round(SSIM_tFM,4)))

axes[2].imshow(InvNet_ind)
MSE_InvNet = np.mean((InvNet_ind - x_truth)**2)
SSIM_InvNet = ssim_batch(InvNet_ind[np.newaxis,np.newaxis,:], x_truth[np.newaxis,np.newaxis,:]).cpu().numpy()
axes[2].title.set_text("InvNet, MSE="+str(np.round(MSE_InvNet,4))+'\n, SSIM='+str(np.round(SSIM_InvNet,4)))

# plt.show()
axes[3].imshow(x_truth)
axes[3].title.set_text("truth")